# Notebook 03 — Feature Engineering

**Project:** Boston Marathon BQ Predictor  
**Author:** Gian Marco  
**Date:** April 2026

## Objectives

1. Remove non-informative or leakage-prone columns
2. Encode categorical variables (Country, Race, Race Category)
3. Create derived features (Is_Home_Country, Age_Squared)
4. Apply target encoding with smoothing and K-Fold for Race (avoiding leakage)
5. Save final datasets ready for modelling
6. Export a reusable preprocessing pipeline to `src/preprocessing.py`

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import KFold
import joblib
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROCESSED_DATA_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

print('Setup ready')

Setup ready


---
## 1. Load data

In [2]:
TRAIN_DIR = Path('../data/train')
TEST_DIR = Path('../data/test')

train = pd.read_csv(TRAIN_DIR / 'train.csv')
test = pd.read_csv(TEST_DIR / 'test.csv')

print(f'Train: {train.shape}')
print(f'Test:  {test.shape}')
print(f'\nCurrent columns: {train.columns.tolist()}')

Train: (225356, 15)
Test:  (74644, 15)

Current columns: ['Year', 'Race', 'Country', 'Gender', 'Age', 'Finish', 'Overall Place', 'Gender Place', 'Race_City', 'Race_State', 'Finishers', 'Category', 'Age Bracket', 'Standard', 'es_BQ']


---
## 2. Remove Leakage-Prone and Redundant Columns

The following columns must NOT be used as features:

- **`Finish`**: final race time, used to compute the target → direct leakage.
- **`Standard`**: BQ qualifying time by age/gender, used to compute the target → direct leakage.
- **`Overall Place`** and **`Gender Place`**: directly derived from finish time → indirect leakage.
- **`Age Bracket`**: redundant with `Age` (discretized version).
- **`Race_City`** and **`Race_State`**: redundant with `Race`.

In [3]:
COLS_TO_DROP = [
    'Finish',           # Leakage: used to build the target
    'Standard',         # Leakage: used to build the target
    'Overall Place',    # Leakage: derived from finish time
    'Gender Place',     # Leakage: derived from finish time
    'Age Bracket',      # Redundant with Age
    'Race_City',        # Redundant with Race
    'Race_State',       # Redundant with Race
]

# If Finish_h exists from EDA, drop it as well
extra_cols = [c for c in ['Finish_h'] if c in train.columns]

train = train.drop(columns=COLS_TO_DROP + extra_cols, errors='ignore')
test = test.drop(columns=COLS_TO_DROP + extra_cols, errors='ignore')

print(f'Train after dropping columns: {train.shape}')
print(f'Remaining columns: {train.columns.tolist()}')

Train after dropping columns: (225356, 8)
Remaining columns: ['Year', 'Race', 'Country', 'Gender', 'Age', 'Finishers', 'Category', 'es_BQ']


---
## 3. Group Low-Frequency Countries

There are more than 200 countries in the dataset, but many have very few runners. A country with only a handful of runners does not provide reliable statistical signal and introduces noise into the model.

We group countries with fewer than 500 runners in the training set into an "Other" category.

**Important:** the threshold is computed using only the training data and then applied to the test set to avoid leakage.

In [4]:
MIN_COUNTRY_COUNT = 500

# Compute frequencies on TRAIN only (never use test for decisions)
country_counts = train['Country'].value_counts()
countries_to_keep = country_counts[country_counts >= MIN_COUNTRY_COUNT].index.tolist()

print(f'Original countries: {train["Country"].nunique()}')
print(f'Countries with more than {MIN_COUNTRY_COUNT} runners: {len(countries_to_keep)}')
print(f'\nCountries kept: {sorted(countries_to_keep)}')

# Apply to train and test
train['Country_grouped'] = train['Country'].where(train['Country'].isin(countries_to_keep), 'Other')
test['Country_grouped'] = test['Country'].where(test['Country'].isin(countries_to_keep), 'Other')

print(f'\nFinal countries in train: {train["Country_grouped"].nunique()}')
print(f'% of runners in Other: {(train["Country_grouped"]=="Other").mean()*100:.2f}%')

Original countries: 163
Countries with more than 500 runners: 15

Countries kept: ['AU', 'BR', 'CA', 'DE', 'ES', 'FR', 'GB', 'IE', 'IT', 'JP', 'MX', 'NL', 'PL', 'TH', 'US']

Final countries in train: 16
% of runners in Other: 4.07%


---
## 4. Derived Feature: `Is_Home_Country`

A runner competing in their home country may have logistical advantages (less jet lag, familiarity with the course, and local climate). This is a simple binary feature but potentially informative.

We compare the runner's `Country` with the country where the race is held. This requires inferring the race country from `Race_State` or the race name (Races.csv).

**Simple strategy:** races with a non-null `Race_State` are assumed to be in the US; the rest are mapped based on race name.

In [5]:
# Reload Races.csv to obtain the country of each race
races_meta = pd.read_csv(PROCESSED_DATA_DIR.parent / 'raw' / 'Races.csv')

# Manual mapping from race name to country code (main non-US races)
race_country_map = {
    'London Marathon': 'GB',
    'Berlin Marathon': 'DE',
    'Tokyo Marathon': 'JP',
    'RnR Madrid Marathon': 'ES',
    'Zurich Marato Barcelona': 'ES',
    'Zurich Maraton Sevilla': 'ES',
    'Zurich Marathon': 'CH',
    'Mexico City Marathon': 'MX',
    'Marathon Pour Tous': 'FR',
    'Athens Marathon': 'GR',
    'Amsterdam Marathon': 'NL',
    'Rotterdam Marathon': 'NL',
    'Valencia Marathon': 'ES',
    'Paris Marathon': 'FR',
    'Copenhagen Marathon': 'DK',
    'Dublin Marathon': 'IE',
    'Stockholm Marathon': 'SE',
    'Vienna City Marathon': 'AT',
    'Cape Town Marathon': 'ZA',
    'Comrades Marathon': 'ZA',
    'Sydney Marathon': 'AU',
    'Melbourne Marathon': 'AU',
    'Toronto Waterfront Marathon': 'CA',
    'Ottawa Marathon': 'CA',
    'LANZAROTE INTERNATIONAL MARATHON': 'ES',
}

def get_race_country(race_name):
    """Assign the country where the race is held. Default: US."""
    return race_country_map.get(race_name, 'US')

train['Race_Country'] = train['Race'].apply(get_race_country)
test['Race_Country'] = test['Race'].apply(get_race_country)

# Create Is_Home_Country feature
train['Is_Home_Country'] = (train['Country'] == train['Race_Country']).astype(int)
test['Is_Home_Country'] = (test['Country'] == test['Race_Country']).astype(int)

print(f'Non-US races mapped: {len(race_country_map)}')
print('Remaining races assumed to be US (majority of the dataset)')

print(f'\n% of runners competing in home country (train): {train["Is_Home_Country"].mean()*100:.2f}%')

print(f'\nBQ rate: home vs away (train):')
print(train.groupby('Is_Home_Country')['es_BQ'].agg(['mean', 'count']).round(4))

Non-US races mapped: 25
Remaining races assumed to be US (majority of the dataset)

% of runners competing in home country (train): 82.79%

BQ rate: home vs away (train):
                   mean   count
Is_Home_Country                
0                0.1875   38777
1                0.1235  186579


---
## 5. Encoding `Race_Category`

The `Category` column from Races.csv contains 4 values: Minor, Moderate, Steep, Very Steep (plus NaN). It is an ordinal variable that captures course difficulty.

We apply a manual ordinal encoding: Minor=0, Moderate=1, Steep=2, Very Steep=3. Missing values (NaN) are filled using the mode computed from the training set.

In [6]:
category_mapping = {
    'Minor': 0,
    'Moderate': 1,
    'Steep': 2,
    'Very Steep': 3,
}

train['Race_Category_enc'] = train['Category'].map(category_mapping)
test['Race_Category_enc'] = test['Category'].map(category_mapping)

# Impute NaN values using the mode from the training set
mode_category = train['Race_Category_enc'].mode()[0]
train['Race_Category_enc'] = train['Race_Category_enc'].fillna(mode_category)
test['Race_Category_enc'] = test['Race_Category_enc'].fillna(mode_category)

print(f'Value used to impute NaN: {mode_category}')
print(f'\nDistribution in train:')
print(train['Race_Category_enc'].value_counts().sort_index())

Value used to impute NaN: 0.0

Distribution in train:
Race_Category_enc
0.0    216765
1.0      2490
2.0      3747
3.0      2354
Name: count, dtype: int64


---
## 6. K-Fold Target Encoding for `Race`

**The problem:** the `Race` variable has ~500 unique values. One-hot encoding would create hundreds of columns (inefficient). A simple target encoding (replacing each race with its average BQ rate) would introduce **leakage**, since the target would be used to construct a feature.

**The solution — K-Fold Target Encoding with Smoothing:**

1. Split the training data into K folds
2. For each fold, compute the average BQ rate per race using ONLY the other K-1 folds
3. Apply this encoding to the current fold
4. For the test set, use the encoding computed on the FULL training set

**Smoothing:** races with few runners in a fold would produce noisy estimates. We apply a weighted average with the global mean based on sample size.

Formula:  
`encoding(race) = (n * mean_race + m * mean_global) / (n + m)`

where:
- `n` = number of samples for that race  
- `m` = smoothing factor

In [7]:
def kfold_target_encode(train_df, test_df, col, target, n_splits=5, smoothing=10, random_state=42):
    """
    K-Fold target encoding to avoid leakage.
    
    Parameters
    ----------
    train_df : DataFrame containing col and target
    test_df  : DataFrame containing col (target not necessarily required)
    col      : name of the categorical column
    target   : name of the target column
    n_splits : number of folds used for encoding
    smoothing: smoothing factor (simple Bayesian smoothing)
    """
    global_mean = train_df[target].mean()
    
    # --- Train encoding (K-Fold) ---
    train_encoded = np.zeros(len(train_df))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(train_df)):
        fold_train = train_df.iloc[train_idx]
        
        # Category-level statistics within the fold
        agg = fold_train.groupby(col)[target].agg(['mean', 'count'])
        
        # Smoothing
        smoothed = (agg['count'] * agg['mean'] + smoothing * global_mean) / (agg['count'] + smoothing)
        
        # Apply encoding to the validation fold
        val_categories = train_df.iloc[val_idx][col]
        train_encoded[val_idx] = val_categories.map(smoothed).fillna(global_mean).values
    
    # --- Test encoding (using the FULL training set) ---
    agg_full = train_df.groupby(col)[target].agg(['mean', 'count'])
    smoothed_full = (agg_full['count'] * agg_full['mean'] + smoothing * global_mean) / (agg_full['count'] + smoothing)
    test_encoded = test_df[col].map(smoothed_full).fillna(global_mean).values
    
    return train_encoded, test_encoded, smoothed_full


# Apply to Race
race_enc_train, race_enc_test, race_encoding_map = kfold_target_encode(
    train, test, col='Race', target='es_BQ', n_splits=5, smoothing=10
)

train['Race_te'] = race_enc_train
test['Race_te'] = race_enc_test

print('Encoding applied to Race')
print(f'\nEncoding range in train: [{train["Race_te"].min():.4f}, {train["Race_te"].max():.4f}]')
print(f'Mean encoding in train: {train["Race_te"].mean():.4f} (should be ~{train["es_BQ"].mean():.4f})')

# Top 10 races with highest encoding (most "BQ-friendly")
print('\nTop 10 races with highest Race_te (full encoding using all training data):')
print(race_encoding_map.sort_values(ascending=False).head(10).round(4))

Encoding applied to Race

Encoding range in train: [0.0045, 0.8543]
Mean encoding in train: 0.1348 (should be ~0.1345)

Top 10 races with highest Race_te (full encoding using all training data):
Race
AbbottWMM Wanda Age Group World Championship    0.8510
Boston Marathon                                 0.5022
Last Chance BQ Chicagoland Marathon             0.4552
Last Chance BQ Grand Rapids                     0.4465
The Light At The End of the Tunnel              0.3998
REVEL Mt. Charleston                            0.3814
REVEL Big Bear                                  0.3716
Mountains 2 Beach Marathon                      0.3280
Tunnel Lite Marathon                            0.3239
California International Marathon               0.3169
dtype: float64


---
## 7. Derived Feature: `Age_Squared`

The relationship between age and BQ probability is not linear. As observed in the EDA, runners aged 60–69 show higher BQ rates than those in the 30–40 range.

A linear model would not capture this pattern using only `Age`. Adding `Age_Squared` allows linear models to model this curvature.

For tree-based models (Random Forest, XGBoost), this feature is redundant, but it does not negatively impact performance.

In [8]:
train['Age_Squared'] = train['Age'] ** 2
test['Age_Squared'] = test['Age'] ** 2

print(f'Age range: [{train["Age"].min()}, {train["Age"].max()}]')
print(f'Age_Squared range: [{train["Age_Squared"].min()}, {train["Age_Squared"].max()}]')

Age range: [18.0, 85.0]
Age_Squared range: [324.0, 7225.0]


---
## 8. One-Hot Encoding of Final Categorical Variables

We now apply simple encoding to the remaining low-cardinality categorical variables:

- `Gender` (M, F) → binary encoding (Male = 1, Female = 0)
- `Country_grouped` (~25–30 values) → one-hot encoding

In [9]:
# Encode Gender as binary
train['Gender_M'] = (train['Gender'] == 'M').astype(int)
test['Gender_M'] = (test['Gender'] == 'M').astype(int)

# One-hot encoding for Country (ensure train and test share the same columns)
train_dummies = pd.get_dummies(train['Country_grouped'], prefix='Country', dtype=int)
test_dummies = pd.get_dummies(test['Country_grouped'], prefix='Country', dtype=int)

# Align columns (test may have fewer categories)
test_dummies = test_dummies.reindex(columns=train_dummies.columns, fill_value=0)

train = pd.concat([train, train_dummies], axis=1)
test = pd.concat([test, test_dummies], axis=1)

print(f'Country one-hot columns created: {len(train_dummies.columns)}')
print(f'Sample: {train_dummies.columns.tolist()[:5]}...')

Country one-hot columns created: 16
Sample: ['Country_AU', 'Country_BR', 'Country_CA', 'Country_DE', 'Country_ES']...


---
## 9. Final Feature Selection and Saving

We remove the original categorical columns that have already been encoded, keeping only the final feature set for modelling.

In [10]:
# Original columns to drop (already encoded)
COLS_ENCODED_DROP = [
    'Race',             # encoded in Race_te
    'Country',          # encoded in Country_grouped and one-hot
    'Country_grouped',  # intermediate, replaced by one-hot
    'Gender',           # encoded in Gender_M
    'Category',         # encoded in Race_Category_enc
    'Race_Country',     # only used to derive Is_Home_Country
    'Finishers',        # race-level info, redundant with Race_te
]

train_final = train.drop(columns=COLS_ENCODED_DROP, errors='ignore')
test_final = test.drop(columns=COLS_ENCODED_DROP, errors='ignore')

# Reorder so that the target is the last column
target_col = 'es_BQ'
feature_cols = [c for c in train_final.columns if c != target_col]
train_final = train_final[feature_cols + [target_col]]
test_final = test_final[feature_cols + [target_col]]

print(f'Train final: {train_final.shape}')
print(f'Test final:  {test_final.shape}')

print(f'\nFinal features ({len(feature_cols)}):')
for c in feature_cols:
    print(f'  - {c} ({train_final[c].dtype})')

# Save datasets
train_final.to_csv(TRAIN_DIR / 'train_features.csv', index=False)
test_final.to_csv(TEST_DIR / 'test_features.csv', index=False)

print('\nSaved files:')
print(f'  {TRAIN_DIR / "train_features.csv"}')
print(f'  {TEST_DIR / "test_features.csv"}')

Train final: (225356, 24)
Test final:  (74644, 24)

Final features (23):
  - Year (int64)
  - Age (float64)
  - Is_Home_Country (int64)
  - Race_Category_enc (float64)
  - Race_te (float64)
  - Age_Squared (float64)
  - Gender_M (int64)
  - Country_AU (int64)
  - Country_BR (int64)
  - Country_CA (int64)
  - Country_DE (int64)
  - Country_ES (int64)
  - Country_FR (int64)
  - Country_GB (int64)
  - Country_IE (int64)
  - Country_IT (int64)
  - Country_JP (int64)
  - Country_MX (int64)
  - Country_NL (int64)
  - Country_Other (int64)
  - Country_PL (int64)
  - Country_TH (int64)
  - Country_US (int64)

Saved files:
  ../data/train/train_features.csv
  ../data/test/test_features.csv


---
## 10. Save Artifacts for Reuse

The `Race` encoding (with smoothing, computed on the full training set) must be saved for use in future predictions (e.g., in the Streamlit app).

In [11]:
artifacts = {
    'countries_to_keep': countries_to_keep,
    'race_country_map': race_country_map,
    'category_mapping': category_mapping,
    'race_encoding_map': race_encoding_map,
    'mode_category': mode_category,
    'global_mean_bq': train['es_BQ'].mean(),
    'feature_cols': feature_cols,
}

joblib.dump(artifacts, MODELS_DIR / 'preprocessing_artifacts.joblib')

print(f'Artifacts saved to {MODELS_DIR / "preprocessing_artifacts.joblib"}')
print(f'\nContents: {list(artifacts.keys())}')

Artifacts saved to ../models/preprocessing_artifacts.joblib

Contents: ['countries_to_keep', 'race_country_map', 'category_mapping', 'race_encoding_map', 'mode_category', 'global_mean_bq', 'feature_cols']


---
## 11. Final Sanity Check

Final validations before closing the notebook:

- No missing values (NaN) in features  
- Correct data types  
- Consistent dataset dimensions  
- Target variable remains unchanged

In [12]:
print('SANITY CHECKS')
print('=' * 60)

# 1. NaN check
nulls_train = train_final.isnull().sum().sum()
nulls_test = test_final.isnull().sum().sum()
print(f'\n1. Missing values in train: {nulls_train}')
print(f'   Missing values in test:  {nulls_test}')
assert nulls_train == 0, 'Missing values found in train'
assert nulls_test == 0, 'Missing values found in test'

# 2. Target consistency
print(f'\n2. % BQ train: {train_final["es_BQ"].mean()*100:.2f}%')
print(f'   % BQ test:  {test_final["es_BQ"].mean()*100:.2f}%')

# 3. Dimensions
print(f'\n3. Train: {train_final.shape[0]:,} rows x {train_final.shape[1]} cols')
print(f'   Test:  {test_final.shape[0]:,} rows x {test_final.shape[1]} cols')
assert train_final.shape[1] == test_final.shape[1], 'Different number of columns'

# 4. Data types (all should be numeric)
non_numeric = train_final.dtypes[~train_final.dtypes.apply(lambda x: np.issubdtype(x, np.number))]
print(f'\n4. Non-numeric columns: {len(non_numeric)}')
if len(non_numeric) > 0:
    print(non_numeric)

# 5. Column consistency
assert list(train_final.columns) == list(test_final.columns), 'Columns do not match'
print(f'\n5. Train/Test columns match: OK')

print('\nAll checks passed. Datasets are ready for modelling in Notebook 04.')

SANITY CHECKS

1. Missing values in train: 0
   Missing values in test:  0

2. % BQ train: 13.45%
   % BQ test:  14.30%

3. Train: 225,356 rows x 24 cols
   Test:  74,644 rows x 24 cols

4. Non-numeric columns: 0

5. Train/Test columns match: OK

All checks passed. Datasets are ready for modelling in Notebook 04.


---
## Notebook 03 Summary

### Final Features Created

| Feature | Type | Description |
|---|---|---|
| `Age` | Numerical | Runner’s age |
| `Age_Squared` | Numerical | Captures non-linearity in the age effect |
| `Year` | Numerical | Used for temporal split |
| `Race_Category_enc` | Ordinal | Course difficulty (0=Minor, 3=Very Steep) |
| `Race_te` | Numerical | Target encoding of Race (K-Fold with smoothing) |
| `Is_Home_Country` | Binary | 1 if the runner competes in their home country |
| `Gender_M` | Binary | 1 if male |
| `Country_*` | Binary | One-hot encoding of Country_grouped (~25–30 categories) |

### Key Decisions

1. **No leakage:** all features derived from finish time were removed  
2. **Country grouping:** threshold of 500, reducing ~222 categories to ~25–30 meaningful ones  
3. **Race K-Fold target encoding:** preserves predictive signal while avoiding leakage  
4. **Age_Squared:** allows linear models to capture the non-linear relationship between age and BQ probability  